# cis100 AGN-on powderday run — corrected (Chabrier) selection

Stages the **AGN-on** radiative transfer for the two CEERS quiescent targets at their correct
epochs (snap 091, z=1.4584 for 000719; snap 096, z=1.2780 for 002962), on a galaxy list selected
in SIMBA's own IMF system.

**Decisions baked into this notebook (2026-07-30):**

1. **The 100 Mpc box, not the 25 Mpc box.** Settled in `box_statistics_cis100_vs_cis25.ipynb`:
   m25 yields 5 analogues per target and **0** AGN hosts (a volume limit — its 4.97e4 cMpc^3
   expects 0.25–0.87 AGN — not physics), *and* it is ~7× more expensive per RT run because
   m25n512 resolves the same M\* with ~8× more particles.
2. **Restage on the Chabrier window.** The existing `rt_selection_{719,2962}.fits` are the
   Salpeter-era tables (median logM\* 10.55 / 10.84, i.e. +0.26 / +0.16 dex above the corrected
   target). CIGALE fitted with `imf=0` (Salpeter) but caesar `masses.stellar` is Chabrier;
   selecting on the Salpeter number is a unit error. It matters more than usual here: 0.25 dex
   in M\* propagates through the M–σ relation into BH mass and hence AGN luminosity — exactly
   the quantity this run measures. New tables are written under **new names**
   (`rt_selection_chab_*.fits`); the Salpeter tables are left untouched as the provenance of the
   finished AGN-off tree.
3. **Reuse what is already computed.** 59 of the 116 corrected-window galaxies already have
   finished AGN-off output in `dusty_simdust`. Only the other 57 need an AGN-off run, so the
   AGN-on/AGN-off differencing costs **116 + 57 = 173** RT runs rather than 232.

**Run tags** (each `run_tag` is a fully isolated directory tree — nothing here overwrites the
finished `dusty_simdust` run):

| tag | what | N |
|---|---|---|
| `dusty_simdust` | finished AGN-off, Salpeter list — **read only**, reused for 59 galaxies | 119 |
| `dusty_simdust_chab_agn` | **new** AGN-on, corrected list | 116 |
| `dusty_simdust_chab_topup` | **new** AGN-off for the 57 corrected-list galaxies not in `dusty_simdust` | 57 |

---

### Two prerequisites, both handled below

**`PartType5` in the cutouts.** `BH_SED=True` builds one point source per black hole from
`PartType5/BH_Mass`, `BH_Mdot`, `Coordinates`. The existing cutouts under
`output/cis100/filtered_particles/snap_{091,096}/` hold only `PartType0`+`PartType4`, so Part 3
**must** re-extract with `EXTRACT_OVERWRITE=True`. This re-reads each ~265 GB snapshot once.
Re-extraction is safe for the finished AGN-off run: PartType5 is purely additive and the
gas/star data are regenerated identically from the same caesar member lists.

**A powderday bug that was fixed on 2026-07-30.** `powderday/front_ends/gadget2pd.py` — the
SIMBA/Gizmo front end — registered the BH fields with malformed names:

```python
ds.add_field(("bh','luminosity"), ...)   # ("bh','luminosity") is ONE STRING
ds.add_field(("bh","luminosity"), ...)   # what source_creation.py indexes
```

yt 4.4's `FieldInfoContainer.add_field` raises `ValueError: Expected name to be a
tuple[str, str]` for a `str`, so **every job would have crashed the moment a black hole was
found** — a failure that only appears once `PartType5` is present, which is why the AGN-off runs
never hit it. All four lines were corrected in both copies (the repo at
`~/powderday/powderday/front_ends/gadget2pd.py`, which wins at runtime because the jobs run
`python ~/powderday/pd_front_end.py` and easy-install.pth has no reorder preamble; and the
installed `powderday-0.1.0-py3.9.egg`). Backups: `*.pre-bhfix-20260730.bak`.
Part 1 re-checks this at run time so a reverted or reinstalled powderday cannot silently
reintroduce it.

> **Latent, not fixed:** line 470 of the same file has the identical typo,
> `ds.add_field(("dust','mass"), ...)`, inside `if cfg.par.dust_grid_type == 'li_ml'`. This
> project runs `dust_grid_type = 'manual'`, so it is dead code here — but it would bite anyone
> switching to the Li+ ML dust model.

## Part 0 · configuration

In [ ]:
import os, glob, sys, re, inspect
import numpy as np
import h5py
from astropy.table import Table
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from simbanator.io.simba import Simulation
from simbanator.sed.makesed import MakeSED

sim = Simulation('cis100')

HOME            = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR          = os.path.join(HOME, 'output', 'cis100', 'analogues_specphot')
SED_OUT         = os.path.join(HOME, 'output', 'cis100', 'sed_analogues')
PARTICLE_PREFIX = 'm100n1024'
hydro_dir_base  = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
PARTITION       = 'INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL'

# ── run tags (isolated trees) ────────────────────────────────────────────────
TAG_AGN        = 'dusty_simdust_chab_agn'     # NEW: AGN-on, corrected list
TAG_OFF_TOPUP  = 'dusty_simdust_chab_topup'   # NEW: AGN-off for the galaxies not already run
TAG_OFF_DONE   = 'dusty_simdust'              # FINISHED AGN-off (Salpeter list) — READ ONLY

PARAMF_AGN     = 'parameters_master-agn.py'   # BH_SED=True, Hopkins, BH_var=False
PARAMF_OFF     = 'parameters_master.py'       # identical except BH_SED=False

SELFILE_AGN    = 'selection_chab_agn.hdf5'
SELFILE_TOPUP  = 'selection_chab_off_topup.hdf5'

# ── selection window, Chabrier (see analogues_specphot_cis100_rt.ipynb Part 0b) ──
# logm    : CIGALE bayes log M* from the imf=1 re-run (NOT the Salpeter 10.54 / 10.92)
# age_gyr : mass-weighted age, MC-propagated from the bayes SFH parameters
TARGETS = {
    '719':  dict(z=1.463, snap=91, z_snap=1.4584, logm=10.29, age_gyr=2.65,
                 dlogm=0.30, dage_gyr=0.85),
    '2962': dict(z=1.266, snap=96, z_snap=1.2780, logm=10.68, age_gyr=3.30,
                 dlogm=0.40, dage_gyr=1.00),
}

PASSIVE_FACTOR       = 0.2
NGAS_MIN, NSTAR_MIN  = 21, 20
EPS_RAD, AGN_FEDD_MIN, AGN_LBOL_MIN = 0.1, 0.02, 1e43
MAX_PER_GROUP        = 40      # cap per (target, AGN class), nearest matches kept

PREFLIGHT_OK = False           # Part 1 sets this; the heavy cells refuse to run without it

print('AGN-on tree      ->', os.path.join(SED_OUT, TAG_AGN))
print('AGN-off top-up   ->', os.path.join(SED_OUT, TAG_OFF_TOPUP))
print('AGN-off finished ->', os.path.join(SED_OUT, TAG_OFF_DONE), '(read only)')

## Part 1 · preflight — will the AGN model actually produce AGN emission?

Cheap, and it must pass before anything heavy runs. Four independent things are checked:

1. **The powderday the Slurm jobs will import** has correctly-formed `("bh", …)` field
   tuples. This is the bug described above; it is invisible until a black hole exists.
2. **`parameters_master-agn.py` really turns the AGN on** — `BH_SED=True`, a BH model, and
   `BH_var=False` so `L_bol = BH_eta · Mdot · c²` follows the SIMBA accretion rates directly
   (keeping the SED consistent with the f_Edd-based AGN classification).
3. **The chosen BH model is self-contained and correctly normalised.** `Hopkins` is a
   tabulated intrinsic quasar template; `Nenkova` would additionally need the CLUMPY torus
   grid (`clumpy_models_201410_tvavg.hdf5`), which is **not installed** — selecting it would
   fail. The template is then evaluated the way `gadget2pd._bhsed_sed` evaluates it —
   `agn_spectrum(log10(L_bol/L_sun))`, last 4 entries dropped — and integrated back over
   `d ln ν`. It must return the `L_bol` it was given; that is the one check that catches an
   erg/s-vs-L_sun slip, which would otherwise silently scale every injected AGN.
4. **The AGN-on and AGN-off masters differ only in `BH_*`.** The AGN-on − AGN-off
   difference only isolates the AGN if dust, grid, photon count and apertures are identical.

> **Never `import powderday` from a notebook.** `powderday/nebular_emission/ASCIItools.py`
> runs, *at import time*,
> ```python
> sys.path.insert(0, sys.argv[1]); par = __import__(sys.argv[2])
> ```
> — it assumes the `argv` of `pd_front_end.py`. In a Jupyter kernel `sys.argv[2]` is the
> `-f` connection-file path, so the import dies with
> `ModuleNotFoundError: No module named '/mnt/home/glorenzon/'`.
> Every check below therefore reads **source text** from the exact files the jobs execute.
> That is stricter than an import would have been: the copy is pinned by path
> (`PD_FRONT_END` from the job template ⇒ `sys.path[0]` for every job) instead of trusting
> whatever this kernel's `sys.path` happens to resolve.

In [ ]:
# ── Part 1 · PREFLIGHT (read-only; NO powderday import — see note above) ─────
import importlib.util as _ilu, zipfile as _zf, contextlib, io

_fail, _warn = [], []
_SEDDIR = os.path.dirname(inspect.getfile(MakeSED))

# 1a. resolve the powderday copy the JOBS will import -------------------------
# The cluster job template hardcodes PD_FRONT_END; python puts that script's
# directory at sys.path[0], so THAT tree is the powderday that executes — the
# installed egg is only a fallback for imports from elsewhere.
_setup = os.path.join(_SEDDIR, 'cosmology_setup_all_cluster.cis.sh')
_m = re.search(r'PD_FRONT_END=\\?"([^"\\]+)', open(_setup).read()) if os.path.exists(_setup) else None
if _m is None:
    _fail.append(f'could not read PD_FRONT_END out of {_setup} — cannot tell which powderday '
                 'the jobs will run, so the BH field-name check would be meaningless.')
    PD_FRONT_END = PD_DIR = None
else:
    PD_FRONT_END = _m.group(1)
    PD_DIR = os.path.dirname(PD_FRONT_END)
    print(f'jobs run          : python {PD_FRONT_END} . parameters_master snapNN_ID')
    print(f'=> sys.path[0]    : {PD_DIR}   (wins over the installed egg)')
    if not os.path.exists(PD_FRONT_END):
        _fail.append(f'{PD_FRONT_END} does not exist')
    _psd = os.path.join(os.environ.get('POWDERDAY_ROOT', os.path.expanduser('~')), 'powderday')
    if os.path.realpath(_psd) != os.path.realpath(PD_DIR):
        _warn.append(f'parameters_master pd_source_dir -> {_psd}, but the jobs import {PD_DIR}; '
                     'filters/templates and code would come from different trees.')

# 1b. the BH field names, as source text --------------------------------------
# The broken form is ("bh','<name>") -- ONE string, not a 2-tuple. yt 4.4 raises
# ValueError("Expected name to be a tuple[str, str]") on it, the moment a BH is found.
# The marker is built by concatenation so the quoting cannot be misread.
_BAD_MARKER = '("bh' + "','"
_WANT = ['coordinates', 'luminosity', 'nu', 'sed']

def _check_bh_names(_src, _label):
    _bad  = _src.count(_BAD_MARKER)
    _good = sorted(set(re.findall(r'add_field\(\("bh","(\w+)"\)', _src)))
    _ok   = (_bad == 0 and _good == _WANT)
    print(f'  {_label:22s} malformed {_bad} | correct {_good} -> {"PASS" if _ok else "FAIL"}')
    return _ok

print('\nBH field names (gadget2pd.py):')
if PD_DIR:
    _g2p_path = os.path.join(PD_DIR, 'powderday', 'front_ends', 'gadget2pd.py')
    if not os.path.exists(_g2p_path):
        _fail.append(f'{_g2p_path} not found')
    elif not _check_bh_names(open(_g2p_path).read(), 'repo (load-bearing)'):
        _fail.append('gadget2pd.py BH field names are malformed — powderday would raise '
                     'ValueError("Expected name to be a tuple[str, str]") as soon as a BH is '
                     'found, i.e. EVERY job would crash. Re-apply the fix '
                     '(backups: *.pre-bhfix-20260730.bak).')

    # the installed egg: only a fallback, so a mismatch is a warning, not a failure
    try:
        _sp = _ilu.find_spec('powderday')        # locates the module; does NOT execute it
        _origin = _sp.origin if _sp else None
    except Exception as _e:
        _origin = None
        _warn.append(f'could not locate an installed powderday: {_e}')
    if _origin and '.egg' + os.sep in _origin:
        _egg = _origin.split('.egg' + os.sep)[0] + '.egg'
        with _zf.ZipFile(_egg) as _z:
            _esrc = _z.read('powderday/front_ends/gadget2pd.py').decode('utf8')
        if not _check_bh_names(_esrc, 'installed egg'):
            _warn.append(f'{os.path.basename(_egg)} still has the malformed names. The jobs use '
                         'the repo copy so they will run, but anything importing powderday from '
                         'outside ~/powderday would crash.')
    elif _origin:
        print(f'  installed copy         {_origin}')

# 1c. the consumer side: source_creation must still index reg["bh","sed"] -----
if PD_DIR:
    _sc_path = os.path.join(PD_DIR, 'powderday', 'source_creation.py')
    _sc_src = open(_sc_path).read().replace("'", '"') if os.path.exists(_sc_path) else ''
    _sc_ok = 'reg["bh","sed"]' in _sc_src
    print(f'  source_creation still indexes reg["bh","sed"]: {_sc_ok}')
    if not _sc_ok:
        _fail.append('source_creation.py no longer indexes reg["bh","sed"] — the field-name '
                     'contract changed; re-derive the expected names before trusting 1b.')

# 2. the AGN parameter master ------------------------------------------------
_pm = os.path.join(_SEDDIR, PARAMF_AGN)
print(f'\nAGN master        : {_pm}')
if not os.path.exists(_pm):
    _fail.append(f'{_pm} not found')
else:
    _cfg = {}
    for _l in open(_pm):
        _m = re.match(r'\s*(BH_SED|BH_eta|BH_var|BH_model|dust_grid_type)\s*=\s*(.+?)\s*(#.*)?$', _l)
        if _m:
            _cfg[_m.group(1)] = _m.group(2).strip()
    for _k in ('BH_SED', 'BH_eta', 'BH_model', 'BH_var', 'dust_grid_type'):
        print(f'  {_k:15s} = {_cfg.get(_k, "(absent)")}')
    if _cfg.get('BH_SED') != 'True':
        _fail.append(f'{PARAMF_AGN}: BH_SED is {_cfg.get("BH_SED")}, must be True')
    if _cfg.get('BH_var') != 'False':
        _fail.append(f'{PARAMF_AGN}: BH_var is {_cfg.get("BH_var")}; True would randomise each '
                     'BH luminosity and break consistency with the f_Edd-based AGN split')

    # 3. the BH model must be installed --------------------------------------
    _model = _cfg.get('BH_model', '').strip('"\'')
    if _model == 'Nenkova':
        _mf = re.search(r'BH_modelfile\s*=.*', open(_pm).read())
        _fail.append('BH_model="Nenkova" needs the CLUMPY grid clumpy_models_201410_tvavg.hdf5, '
                     'which is not installed on this cluster. Use "Hopkins".')
    elif PD_DIR:
        # hopkins.py imports nothing but numpy, so it can be loaded straight from
        # its file — bypassing powderday/__init__.py and the ASCIItools argv trap.
        # It is chatty, hence the redirect_stdout.
        _hop = os.path.join(PD_DIR, 'powderday', 'agn_models', 'hopkins.py')
        try:
            _s = _ilu.spec_from_file_location('_pd_hopkins_standalone', _hop)
            _hmod = _ilu.module_from_spec(_s)
            _s.loader.exec_module(_hmod)

            # gadget2pd._bhsed_sed calls agn_spectrum(log10(L_bol/Lsun)) and drops the
            # last 4 entries (band luminosities, not SED points); the second column is
            # log10(nu*L_nu / [erg/s]). Reproduce that exactly, at the AGN threshold
            # luminosity, and integrate it back: sum(nu*L_nu dlnnu) must return L_bol.
            # A units slip anywhere in that chain shows up here as orders of magnitude.
            _LSUN = 3.9e33                                   # erg/s, as the template defines it
            with contextlib.redirect_stdout(io.StringIO()):
                _lognu, _lognulnu = _hmod.agn_spectrum(np.log10(AGN_LBOL_MIN / _LSUN))
            _lognu   = np.asarray(_lognu)[:-4]
            _nulnu   = 10.0 ** np.asarray(_lognulnu)[:-4]
            _trap    = getattr(np, 'trapezoid', None) or np.trapz   # renamed in NumPy 2.0
            _lrec    = np.log(10) * _trap(_nulnu, _lognu)     # erg/s
            _peak_um = 2.998e14 / 10 ** float(_lognu[np.argmax(_nulnu)])
            print(f'\n  Hopkins template OK: {len(_lognu)} SED points, '
                  f'{2.998e14 / 10 ** _lognu.max():.3g}-{2.998e14 / 10 ** _lognu.min():.3g} um')
            print(f'    L_bol = {AGN_LBOL_MIN:.1e} erg/s in -> {_lrec:.3e} erg/s integrated back '
                  f'(ratio {_lrec / AGN_LBOL_MIN:.2f}; <1 because the grid stops at '
                  f'300 um / 100 keV), peak at {_peak_um:.3g} um')
            if not 0.5 < _lrec / AGN_LBOL_MIN < 1.5:
                _fail.append(f'the Hopkins template returns {_lrec / AGN_LBOL_MIN:.3g}x the '
                             'bolometric luminosity it was given — the erg/s vs Lsun convention '
                             'has changed, so injected AGN luminosities would be wrong.')
        except Exception as _e:
            _fail.append(f'could not evaluate the Hopkins template at {_hop}: {_e!r}')

# 4. AGN-on vs AGN-off master: only BH_* settings may differ -----------------
# The AGN-on minus AGN-off difference is only interpretable as "the AGN" if the two
# runs are identical in every other respect (dust, grid, photon count, apertures,
# filters). Differences confined to BH_* keys are fine — BH_model/BH_var/BH_modelfile
# are inert when BH_SED=False. Anything else contaminates the differencing.
_pm_off = os.path.join(_SEDDIR, PARAMF_OFF)
if os.path.exists(_pm_off):
    _a = [l for l in open(_pm).read().splitlines() if l.strip() and not l.strip().startswith('#')]
    _b = [l for l in open(_pm_off).read().splitlines() if l.strip() and not l.strip().startswith('#')]
    _diff = sorted(set(_a) ^ set(_b))
    _keys = set()
    for _l in _diff:
        _mk = re.match(r'\s*(\w+)\s*=', _l)
        _keys.add(_mk.group(1) if _mk else _l.strip())
    _nonbh = sorted(_k for _k in _keys if not _k.startswith('BH_'))
    print(f'\nAGN-on vs AGN-off master: {len(_diff)} differing non-comment lines, '
          f'keys {sorted(_keys)}')
    for _l in _diff:
        print('   ', _l.strip())
    if _nonbh:
        _fail.append(f'the AGN-on and AGN-off masters differ outside the BH_* settings '
                     f'({_nonbh}). The AGN-on minus AGN-off difference would then not isolate '
                     'the AGN — make everything except BH_* identical first.')
    else:
        print('   -> differences are confined to BH_* (BH_model/BH_var are inert when '
              'BH_SED=False): the AGN-off run is a clean control.')

print('\n' + '=' * 78)
for _w in _warn:
    print('WARN:', _w)
if _fail:
    PREFLIGHT_OK = False
    for _f in _fail:
        print('FAIL:', _f)
    raise RuntimeError('Preflight failed — do NOT run the extraction or staging cells.')
PREFLIGHT_OK = True
print('PREFLIGHT PASSED — the AGN model is wired up and will emit.')

## Part 2 · build the corrected (Chabrier) selection

Identical cuts to `analogues_specphot_cis100_rt.ipynb` Part 1 — mass window → mass-weighted age
window → quiescent (sSFR < 0.2/t_H) → resolution floors → nearest-match cap of 40 per
(target, AGN class) — but centred on the Chabrier masses. Written to `rt_selection_chab_*.fits`
so the Salpeter tables that document the finished AGN-off run stay intact.

In [ ]:
# ── Part 2 · corrected selection (writes NEW files; old tables untouched) ────
SELECTED = {}
for tid, T in TARGETS.items():
    with h5py.File(sim.get_caesar_file(T['snap']), 'r') as f:
        d = f['galaxy_data']
        gid  = np.asarray(d['GroupID'][:], int)
        ms   = np.asarray(d['dicts/masses.stellar'][:], float)
        age  = np.asarray(d['dicts/ages.mass_weighted'][:], float)
        md_  = np.asarray(d['dicts/masses.dust'][:], float)
        mgas = np.asarray(d['dicts/masses.gas'][:], float)
        sfr  = np.asarray(d['sfr'][:], float)
        ngas = np.asarray(d['ngas'][:], int)
        nstar= np.asarray(d['nstar'][:], int)
        cen  = np.asarray(d['central'][:], int)
        fedd = np.asarray(d['bh_fedd'][:], float)
        mdot = np.asarray(d['bhmdot'][:], float)
        r50s = np.asarray(d['dicts/radii.stellar_half_mass'][:], float)

    with np.errstate(invalid='ignore', divide='ignore'):
        lm   = np.log10(np.where(ms > 0, ms, np.nan))
        ssfr = sfr / ms
        lbol = (EPS_RAD * (mdot * u.Msun / u.yr) * const.c**2).to(u.erg / u.s).value
    agn = (np.nan_to_num(fedd) >= AGN_FEDD_MIN) & (np.nan_to_num(lbol) >= AGN_LBOL_MIN)
    tH  = Planck13.age(T['z_snap']).to(u.yr).value

    m_m = np.abs(lm - T['logm']) <= T['dlogm']
    m_a = np.abs(age - T['age_gyr']) <= T['dage_gyr']
    m_q = ssfr < PASSIVE_FACTOR / tH
    m_r = (ngas >= NGAS_MIN) & (nstar >= NSTAR_MIN)
    sel = m_m & m_a & m_q & m_r
    print(f"── {tid}  snap {T['snap']} (z={T['z_snap']})  window logM*="
          f"{T['logm']:.2f}±{T['dlogm']:.2f} [Chabrier], age={T['age_gyr']:.2f}±{T['dage_gyr']:.2f} ──")
    print(f"  mass window          : {int(m_m.sum()):5d}")
    print(f"  + age window         : {int((m_m & m_a).sum()):5d}")
    print(f"  + quiescent          : {int((m_m & m_a & m_q).sum()):5d}")
    print(f"  + resolution floors  : {int(sel.sum()):5d}"
          f"   -> AGN {int((sel & agn).sum())} | non-AGN {int((sel & ~agn).sum())}")

    dist = np.sqrt(((lm - T['logm']) / T['dlogm'])**2 + ((age - T['age_gyr']) / T['dage_gyr'])**2)
    keep = np.zeros(sel.size, bool)
    for cls_mask in (sel & agn, sel & ~agn):
        ii = np.where(cls_mask)[0]
        if MAX_PER_GROUP is not None and ii.size > MAX_PER_GROUP:
            ii = ii[np.argsort(dist[ii])[:MAX_PER_GROUP]]
        keep[ii] = True
    print(f"  staged (cap {MAX_PER_GROUP}/group): {int(keep.sum())}"
          f"   -> AGN {int((keep & agn).sum())} | non-AGN {int((keep & ~agn).sum())}")

    A = Table(dict(SNAPSHOT=np.full(keep.sum(), T['snap']), GROUPID_SNAPSHOT=gid[keep],
                   REDSHIFT=np.full(keep.sum(), T['z_snap']),
                   LOG_MSTAR=lm[keep], AGE_STAR=age[keep], SSFR=ssfr[keep],
                   MGAS=mgas[keep], MDUST=md_[keep],
                   LOG_MDUST=np.log10(np.where(md_[keep] > 0, md_[keep], np.nan)),
                   NGAS=ngas[keep], NSTAR=nstar[keep], CENTRAL=cen[keep],
                   FEDD=fedd[keep], LBOL=lbol[keep], IS_AGN=agn[keep],
                   R50_STAR=r50s[keep], DIST_MATCH=dist[keep]))
    A['AGN_CLASS'] = np.where(np.asarray(A['IS_AGN']), 'AGN', 'no_AGN')
    A.sort('GROUPID_SNAPSHOT')
    out = os.path.join(OUTDIR, f'rt_selection_chab_{tid}.fits')
    A.write(out, overwrite=True)
    SELECTED[tid] = A
    print(f'  median logM* = {np.median(A["LOG_MSTAR"]):.3f}  (target {T["logm"]:.2f})  -> {out}\n')

_pairs = np.unique(np.concatenate([np.stack([np.asarray(A['SNAPSHOT'], int),
                                             np.asarray(A['GROUPID_SNAPSHOT'], int)], axis=1)
                                   for A in SELECTED.values()]), axis=0)
SNAPS, IDS = _pairs[:, 0], _pairs[:, 1]
print(f'total unique RT targets: {len(SNAPS)} over snaps {sorted(set(SNAPS.tolist()))}')

## Part 3 · reconcile against the finished AGN-off tree

Which of the corrected-window galaxies already have a usable AGN-off SED, and which need one.
The reused outputs were produced with the same `parameters_master.py`, the same snapshots and
the same caesar member lists, so they are physically identical to a fresh run — only the
*selection* that picked them differed.

In [ ]:
# ── Part 3 · what already exists in dusty_simdust ────────────────────────────
def _rtout(tag, snap, gid):
    return os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{snap:03d}',
                        f'gal_{int(gid)}', f'snap{snap:03d}.galaxy{int(gid):06d}.rtout.sed')

REUSE, TOPUP = [], []
for snap, gid in zip(SNAPS, IDS):
    (REUSE if os.path.exists(_rtout(TAG_OFF_DONE, snap, gid)) else TOPUP).append((int(snap), int(gid)))

print(f'corrected-window galaxies      : {len(SNAPS)}')
print(f'  AGN-off already finished     : {len(REUSE):3d}  (reuse from {TAG_OFF_DONE})')
print(f'  AGN-off still needed         : {len(TOPUP):3d}  (-> {TAG_OFF_TOPUP})')
print(f'  AGN-on to run                : {len(SNAPS):3d}  (-> {TAG_AGN})')
print(f'  TOTAL new RT runs            : {len(SNAPS) + len(TOPUP):3d}')

for _sn in sorted(set(int(s) for s in SNAPS)):
    _n  = sum(1 for s, g in zip(SNAPS, IDS) if int(s) == _sn)
    _nr = sum(1 for s, g in REUSE if s == _sn)
    print(f'   snap {_sn:03d}: {_n:3d} galaxies | reuse {_nr:3d} | top-up {_n - _nr:3d}')

TOPUP_SNAPS = np.array([s for s, g in TOPUP], int)
TOPUP_IDS   = np.array([g for s, g in TOPUP], int)

# record the reuse map so the analysis notebook knows where each AGN-off SED lives
_rows = [(s, g, TAG_OFF_DONE) for s, g in REUSE] + [(s, g, TAG_OFF_TOPUP) for s, g in TOPUP]
_rows.sort()
_map = Table(rows=_rows, names=('SNAPSHOT', 'GROUPID_SNAPSHOT', 'AGNOFF_TAG'))
_mapout = os.path.join(OUTDIR, 'agnoff_source_map_chab.fits')
_map.write(_mapout, overwrite=True)
print(f'\nAGN-off provenance map -> {_mapout}')

## Part 4 · re-extract the particle cutouts **including `PartType5`**  ⚠️ heavy

Reads each ~265 GB snapshot once. `EXTRACT_OVERWRITE=True` is required — `extract_particles`
skips files that already exist, so without it the existing `PartType0`+`PartType4` cutouts would
survive untouched and the whole AGN run would silently find no black holes.

`extract_particles` copies **every** dataset of each requested particle type (`ignore_fields`
defaults to empty), so `BH_Mass` and `BH_Mdot` — the two fields the front end needs — come along
automatically. Galaxy membership for PartType5 comes from caesar's `bhlist`.

In [ ]:
# ── Part 4 · particle re-extraction (HEAVY — run on the cluster) ─────────────
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight did not pass — fix that before spending ~530 GB of reads.')

from simbanator.analysis import extract_particles

EXTRACT_OVERWRITE = True                                   # MUST be True: adds PartType5
EXTRACT_PTYPES    = ("PartType0", "PartType4", "PartType5")

for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    _cs = sim.load_catalog(snap=_snap)
    extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, ptypes=EXTRACT_PTYPES,
                      sim_name=sim.name, prefix=PARTICLE_PREFIX,
                      overwrite=EXTRACT_OVERWRITE, verbose=1)
    del _cs
print('\nparticle re-extraction complete ->', hydro_dir_base)

### Part 4b · verify the black holes actually landed

A cutout with no `PartType5` is legitimate — not every galaxy hosts a black hole — but the count
must match caesar's `nbh`, and `BH_Mass`/`BH_Mdot` must be present, or the run is AGN-free.

In [ ]:
# ── Part 4b · verify PartType5 (cheap; do not skip) ──────────────────────────
_nbh_caesar = {}
for _sn in sorted(set(int(s) for s in SNAPS)):
    with h5py.File(sim.get_caesar_file(_sn), 'r') as f:
        _nbh_caesar[_sn] = dict(zip(np.asarray(f['galaxy_data/GroupID'][:], int),
                                    np.asarray(f['galaxy_data/nbh'][:], int)))

n_tot = n_with_bh = n_bh_particles = 0
missing, mismatch, no_fields = [], [], []
for _snap, _gid in zip(SNAPS, IDS):
    _snap, _gid = int(_snap), int(_gid)
    p = os.path.join(hydro_dir_base, f'snap_{_snap:03d}',
                     f'{PARTICLE_PREFIX}_snap{_snap:03d}_gal{_gid:06d}.h5')
    if not os.path.exists(p):
        missing.append((_snap, _gid)); continue
    n_tot += 1
    with h5py.File(p, 'r') as f:
        n_here = f['PartType5/BH_Mass'].shape[0] if 'PartType5' in f and 'BH_Mass' in f['PartType5'] else 0
        if 'PartType5' in f:
            if not {'BH_Mass', 'BH_Mdot', 'Coordinates'} <= set(f['PartType5'].keys()):
                no_fields.append((_snap, _gid, sorted(f['PartType5'].keys())))
            if n_here:
                n_with_bh += 1; n_bh_particles += n_here
    if n_here != _nbh_caesar[_snap].get(_gid, 0):
        mismatch.append((_snap, _gid, n_here, _nbh_caesar[_snap].get(_gid, 0)))

print(f'cutouts found        : {n_tot} / {len(SNAPS)}')
print(f'  with >=1 black hole: {n_with_bh}')
print(f'  total BH particles : {n_bh_particles}')
if missing:   print(f'  !! MISSING cutouts : {len(missing)} e.g. {missing[:5]}')
if no_fields: print(f'  !! PartType5 missing BH_Mass/BH_Mdot/Coordinates: {no_fields[:5]}')
if mismatch:  print(f'  !! nbh != caesar   : {len(mismatch)} e.g. {mismatch[:5]}')

if n_with_bh == 0:
    raise RuntimeError('No cutout contains a black hole — re-run Part 4 with '
                       'EXTRACT_OVERWRITE=True. Staging now would reproduce an AGN-free run.')
if missing or no_fields or mismatch:
    raise RuntimeError('Cutouts are not consistent with caesar — resolve before staging.')

# how many of the AGN-classified hosts actually carry a BH particle
for tid, A in SELECTED.items():
    _isagn = np.asarray(A['IS_AGN'], bool)
    print(f'  {tid}: IS_AGN hosts = {int(_isagn.sum())} / {len(A)}')
print('\nPartType5 verified — safe to stage.')

## Part 5 · stage the runs (job files written, **not** submitted)

Two independent trees. Neither touches `dusty_simdust`.

In [ ]:
# ── Part 5a · AGN-ON run, all 116 corrected-window galaxies ──────────────────
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight did not pass.')

ms_agn = MakeSED(sim, nnodes=1, model_run_name=TAG_AGN,
                 hydro_dir_base=hydro_dir_base, selection_file=SELFILE_AGN,
                 output_dir=SED_OUT, run_tag=TAG_AGN)
ms_agn.selection_gals(snaps=SNAPS, galaxyID=IDS)
ms_agn.create_master('cluster', 'plist', radius=None, partition=PARTITION,
                     prefix=PARTICLE_PREFIX, paramf=PARAMF_AGN, snaps_to_run=None)
print('staged AGN-on ->', os.path.join(SED_OUT, TAG_AGN))

In [ ]:
# ── Part 5b · AGN-OFF top-up, only the galaxies without existing output ──────
if len(TOPUP_IDS) == 0:
    print('nothing to top up — every corrected-window galaxy already has an AGN-off SED')
else:
    ms_off = MakeSED(sim, nnodes=1, model_run_name=TAG_OFF_TOPUP,
                     hydro_dir_base=hydro_dir_base, selection_file=SELFILE_TOPUP,
                     output_dir=SED_OUT, run_tag=TAG_OFF_TOPUP)
    ms_off.selection_gals(snaps=TOPUP_SNAPS, galaxyID=TOPUP_IDS)
    ms_off.create_master('cluster', 'plist', radius=None, partition=PARTITION,
                         prefix=PARTICLE_PREFIX, paramf=PARAMF_OFF, snaps_to_run=None)
    print('staged AGN-off top-up ->', os.path.join(SED_OUT, TAG_OFF_TOPUP))

### Part 5c · confirm each tree points at the master it should

The single most expensive mistake available here is staging the AGN tree against the AGN-*off*
master (or vice versa) and discovering it only after the RT finishes. `create_master` copies the
chosen `paramf` into each `snap_*` directory as `parameters_master.py`, so the staged copy is
what actually runs — check that, not the source file.

In [ ]:
# ── Part 5c · verify the staged parameter files ──────────────────────────────
def _check(tag, want_bh):
    print(f'\n=== {tag} (expect BH_SED = {want_bh}) ===')
    ok = True
    for _snap in sorted(set(int(s) for s in SNAPS)):
        jdir = os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{_snap:03d}')
        if not os.path.isdir(jdir):
            jdir = os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{_snap}')
        if not os.path.isdir(jdir):
            print(f'  snap {_snap}: no staged directory'); continue
        pm = os.path.join(jdir, 'parameters_master.py')
        if not os.path.exists(pm):
            print(f'  snap {_snap}: !! no parameters_master.py written'); ok = False; continue
        vals = {}
        for l in open(pm):
            m = re.match(r'\s*(BH_SED|BH_model|BH_var|dust_grid_type)\s*=\s*(\S+)', l)
            if m: vals[m.group(1)] = m.group(2)
        nids = sum(1 for _ in open(os.path.join(jdir, 'ids.txt'))) \
               if os.path.exists(os.path.join(jdir, 'ids.txt')) else 0
        print(f'  snap {_snap}: {vals}   ids.txt = {nids}')
        if vals.get('BH_SED') != str(want_bh):
            print(f'     !! BH_SED is {vals.get("BH_SED")}, expected {want_bh} — wrong paramf'); ok = False
    return ok

_a = _check(TAG_AGN, True)
_b = _check(TAG_OFF_TOPUP, False) if len(TOPUP_IDS) else True
print('\n' + ('staging verified' if (_a and _b) else '!! STAGING PROBLEM — do not submit'))

print('\n── submit with ──')
for tag in ([TAG_AGN] + ([TAG_OFF_TOPUP] if len(TOPUP_IDS) else [])):
    for _snap in sorted(set(int(s) for s in SNAPS)):
        jdir = os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{_snap:03d}')
        if not os.path.isdir(jdir):
            jdir = os.path.join(SED_OUT, tag, 'powderday_sed_out', f'snap_{_snap}')
        jobs = sorted(glob.glob(os.path.join(jdir, 'master.snap*.job')))
        if jobs:
            print(f'cd {jdir} && sbatch {os.path.basename(jobs[-1])}')
        elif os.path.isdir(jdir):
            print(f'!! no master.snap*.job in {jdir} — staging failed?')

## Part 6 · after the RT finishes

**1. Completeness.** Expect one `.rtout.sed` per galaxy in each tree:

```bash
ls output/cis100/sed_analogues/dusty_simdust_chab_agn/powderday_sed_out/snap_*/gal_*/*.rtout.sed | wc -l
ls output/cis100/sed_analogues/dusty_simdust_chab_topup/powderday_sed_out/snap_*/gal_*/*.rtout.sed | wc -l
```

**2. Confirm the AGN actually emitted — this is the check that matters.** `BH_SED=True` alone
proves nothing; the cis25 run carried it and produced no AGN emission at all. Two independent
signatures:

```bash
# dump_AGN_SEDs writes this whenever nholes > 0
ls output/cis100/sed_analogues/dusty_simdust_chab_agn/powderday_sed_out/snap_*/gal_*/bh_sed*.npz | wc -l
# and the log must NOT say the BHs were never found
grep -l "BH source creation failed" output/cis100/sed_analogues/dusty_simdust_chab_agn/powderday_sed_out/snap_*/gal_*/*.LOG | wc -l
grep -h "Number AGNs in the cutout" output/cis100/sed_analogues/dusty_simdust_chab_agn/powderday_sed_out/snap_*/gal_*/*.LOG | sort | uniq -c
```

In the rtout files an AGN appears as a **`point`** source (`m.add_point_source`), alongside the
stellar `point_collection`. A tree with only `point_collection` sources has no AGN in it.

**3. Difference the trees.** `F_agn_on − F_agn_off` per galaxy, taking each galaxy's AGN-off SED
from the tag recorded in `agnoff_source_map_chab.fits` (`dusty_simdust` for the 59 reused,
`dusty_simdust_chab_topup` for the rest). Only the `IS_AGN` hosts should move; the non-AGN hosts
are the built-in control and their on/off difference should be ≈0. A non-zero difference there
is itself worth reporting — it would mean low-f_Edd black holes are still radiating, and that an
L_bol-only cut would reclassify them.

**4. Then** the 002962 mid-IR comparison can finally use the CIGALE **total** curve rather than
the galaxy-only one, because the simulated SED now contains a torus.

---

**Convergence — expected, and not a problem.** Every RT output will carry `converged = 'no'`,
stopped at the 7-iteration cap in `set_n_initial_iterations(7)`. Verified against the AGN-off run
logs: the metric is flat at 1.07–1.24 against the 1.01 threshold with `Difference from previous
iteration: 1.00` (zero improvement) from iteration 2 onward, plus 45 `specific_energy below
minimum allowed` warnings per log. It is a Monte-Carlo photon noise floor in the sparsest
outskirt cells of the ±100 kpc box, and the criterion is the worst 1% of cells — negligible dust
mass, negligible SED contribution. **Do not raise the iteration cap.** The lever, if one is ever
needed, is `n_photons_initial` (currently 1e7).

**Known caveat carried forward.** Neither box reaches the target mass-weighted age: for 002962
(target 3.30 Gyr) the oldest quiescent galaxy in the mass window is 3.21 Gyr, and 0 of 567 reach
target; for 000719 (target 2.65 Gyr) only 4 of 330 do. SIMBA does not make quiescent galaxies
this old at this mass and epoch — report it, do not tune the window.